In [1]:
from pyspark.sql import functions as F
from datetime import date, timedelta

# =========================================================
# 1. Parâmetros
# =========================================================

ano_inicio = 2009
ano_fim = 2030

data_inicio = f"{ano_inicio}-01-01"
data_fim = f"{ano_fim}-12-31"


# =========================================================
# 2. Função para calcular a Páscoa
# Algoritmo de Meeus/Jones/Butcher para calendário gregoriano
# =========================================================

def calcula_pascoa(ano):
    a = ano % 19
    b = ano // 100
    c = ano % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = ((h + l - 7 * m + 114) % 31) + 1
    return date(ano, mes, dia)


# =========================================================
# 3. Monta lista de feriados
# =========================================================

feriados = []

for ano in range(ano_inicio, ano_fim + 1):

    pascoa = calcula_pascoa(ano)

    # Feriados nacionais fixos
    feriados.extend([
        (date(ano, 1, 1),   "Confraternização Universal", "Nacional"),
        (date(ano, 4, 21),  "Tiradentes", "Nacional"),
        (date(ano, 5, 1),   "Dia do Trabalho", "Nacional"),
        (date(ano, 9, 7),   "Independência do Brasil", "Nacional"),
        (date(ano, 10, 12), "Nossa Senhora Aparecida", "Nacional"),
        (date(ano, 11, 2),  "Finados", "Nacional"),
        (date(ano, 11, 15), "Proclamação da República", "Nacional"),
        (date(ano, 11, 20), "Consciência Negra", "Nacional"),
        (date(ano, 12, 25), "Natal", "Nacional"),
    ])
    
    # Datas especiais / vésperas
    feriados.extend([
        (date(ano, 12, 24), "Véspera de Natal", "Véspera / ponto facultativo"),
        (date(ano, 12, 31), "Véspera de Ano Novo", "Véspera / ponto facultativo"),
    ])

    # Feriados móveis
    feriados.extend([
        (pascoa - timedelta(days=48), "Carnaval - segunda-feira", "Nacional/Facultativo"),
        (pascoa - timedelta(days=47), "Carnaval - terça-feira", "Nacional/Facultativo"),
        (pascoa - timedelta(days=46), "Quarta-feira de Cinzas", "Nacional/Facultativo"),
        (pascoa - timedelta(days=2),  "Sexta-feira Santa", "Nacional"),
        (pascoa,                      "Páscoa", "Nacional"),
        (pascoa + timedelta(days=60), "Corpus Christi", "Nacional"),
    ])

    # São Paulo capital / estado
    feriados.extend([
        (date(ano, 1, 25), "Aniversário da cidade de São Paulo", "Municipal - São Paulo"),
        (date(ano, 7, 9),  "Revolução Constitucionalista", "Estadual - São Paulo"),
    ])


# Remove duplicados, preservando nomes combinados se houver mais de um feriado na data
feriados_dict = {}

for data_feriado, nome, abrangencia in feriados:
    if data_feriado not in feriados_dict:
        feriados_dict[data_feriado] = {
            "nomes": set(),
            "abrangencias": set()
        }
    feriados_dict[data_feriado]["nomes"].add(nome)
    feriados_dict[data_feriado]["abrangencias"].add(abrangencia)

feriados_final = [
    (
        d.isoformat(),
        " | ".join(sorted(v["nomes"])),
        " | ".join(sorted(v["abrangencias"]))
    )
    for d, v in feriados_dict.items()
]

df_feriados = spark.createDataFrame(
    feriados_final,
    ["data", "nome_feriado", "tipo_feriado"]
).withColumn(
    "data",
    F.to_date("data")
)


# =========================================================
# 4. Cria calendário base
# =========================================================

df_cal = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{data_inicio}'),
                to_date('{data_fim}'),
                interval 1 day
            )
        ) AS data
    """)
    .withColumn("ano", F.year("data"))
    .withColumn("mes_num", F.month("data"))
    .withColumn("dia_mes", F.dayofmonth("data"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("semana_ano", F.weekofyear("data"))
    .withColumn("data_inicio_mes", F.trunc("data", "MM"))
    .withColumn("data_fim_mes", F.last_day("data"))
)


# =========================================================
# 5. Semana começando na segunda-feira
# Spark dayofweek:
# domingo = 1, segunda = 2, ..., sábado = 7
# =========================================================

df_cal = (
    df_cal
    .withColumn("dia_semana_num_spark", F.dayofweek("data"))
    .withColumn(
        "dia_semana_num",
        F.expr("""
            CASE dayofweek(data)
                WHEN 2 THEN 1
                WHEN 3 THEN 2
                WHEN 4 THEN 3
                WHEN 5 THEN 4
                WHEN 6 THEN 5
                WHEN 7 THEN 6
                WHEN 1 THEN 7
            END
        """)
    )
    .withColumn(
        "data_inicio_semana",
        F.date_sub("data", F.col("dia_semana_num") - 1)
    )
    .withColumn(
        "data_fim_semana",
        F.date_add("data_inicio_semana", 6)
    )
)


# =========================================================
# 6. Nomes em português
# =========================================================

df_cal = (
    df_cal
    .withColumn(
        "nome_mes",
        F.expr("""
            CASE mes_num
                WHEN 1 THEN 'janeiro'
                WHEN 2 THEN 'fevereiro'
                WHEN 3 THEN 'março'
                WHEN 4 THEN 'abril'
                WHEN 5 THEN 'maio'
                WHEN 6 THEN 'junho'
                WHEN 7 THEN 'julho'
                WHEN 8 THEN 'agosto'
                WHEN 9 THEN 'setembro'
                WHEN 10 THEN 'outubro'
                WHEN 11 THEN 'novembro'
                WHEN 12 THEN 'dezembro'
            END
        """)
    )
    .withColumn("nome_mes_3", F.substring("nome_mes", 1, 3))
    .withColumn(
        "nome_dia_semana",
        F.expr("""
            CASE dia_semana_num
                WHEN 1 THEN 'segunda-feira'
                WHEN 2 THEN 'terça-feira'
                WHEN 3 THEN 'quarta-feira'
                WHEN 4 THEN 'quinta-feira'
                WHEN 5 THEN 'sexta-feira'
                WHEN 6 THEN 'sábado'
                WHEN 7 THEN 'domingo'
            END
        """)
    )
    .withColumn(
        "nome_dia_3",
        F.expr("""
            CASE dia_semana_num
                WHEN 1 THEN 'seg'
                WHEN 2 THEN 'ter'
                WHEN 3 THEN 'qua'
                WHEN 4 THEN 'qui'
                WHEN 5 THEN 'sex'
                WHEN 6 THEN 'sáb'
                WHEN 7 THEN 'dom'
            END
        """)
    )
)


# =========================================================
# 7. Junta feriados
# =========================================================

df_cal = (
    df_cal
    .join(df_feriados, on="data", how="left")
    .withColumn(
        "eh_feriado",
        F.when(F.col("nome_feriado").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "eh_dia_util",
        F.when(
            (F.col("dia_semana_num").between(1, 5)) & (F.col("eh_feriado") == 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)


# =========================================================
# 8. Lógica de emenda
# =========================================================
# Conceito usado:
# - segunda-feira útil antes de feriado na terça;
# - sexta-feira útil depois de feriado na quinta;
# - dia útil entre feriado e fim de semana;
# - dia útil entre fim de semana e feriado.
# =========================================================

df_flags = df_cal.select(
    F.col("data").alias("data_ref"),
    F.col("eh_feriado").alias("eh_feriado_ref")
)

df_cal = (
    df_cal
    .join(
        df_flags.withColumnRenamed("data_ref", "data_anterior")
                .withColumnRenamed("eh_feriado_ref", "feriado_dia_anterior"),
        df_cal["data"] == F.date_add(F.col("data_anterior"), 1),
        "left"
    )
    .drop("data_anterior")
    .join(
        df_flags.withColumnRenamed("data_ref", "data_posterior")
                .withColumnRenamed("eh_feriado_ref", "feriado_dia_posterior"),
        df_cal["data"] == F.date_sub(F.col("data_posterior"), 1),
        "left"
    )
    .drop("data_posterior")
)

df_cal = (
    df_cal
    .withColumn("feriado_dia_anterior", F.coalesce("feriado_dia_anterior", F.lit(0)))
    .withColumn("feriado_dia_posterior", F.coalesce("feriado_dia_posterior", F.lit(0)))
    .withColumn(
        "eh_emenda",
        F.when(
            # Segunda antes de feriado na terça
            (F.col("dia_semana_num") == 1) & (F.col("feriado_dia_posterior") == 1) & (F.col("eh_feriado") == 0),
            F.lit(1)
        ).when(
            # Sexta depois de feriado na quinta
            (F.col("dia_semana_num") == 5) & (F.col("feriado_dia_anterior") == 1) & (F.col("eh_feriado") == 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "tipo_emenda",
        F.when(
            (F.col("dia_semana_num") == 1) & (F.col("feriado_dia_posterior") == 1) & (F.col("eh_feriado") == 0),
            F.lit("segunda antes de feriado na terça")
        ).when(
            (F.col("dia_semana_num") == 5) & (F.col("feriado_dia_anterior") == 1) & (F.col("eh_feriado") == 0),
            F.lit("sexta depois de feriado na quinta")
        ).otherwise(F.lit(None))
    )
)


# =========================================================
# 9. Colunas auxiliares úteis para Power BI
# =========================================================

df_cal = (
    df_cal
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("ano_mes_num", F.expr("ano * 100 + mes_num"))
    .withColumn("mes_ano_nome", F.concat_ws("/", F.col("nome_mes_3"), F.col("ano")))
    .withColumn("data_int", F.date_format("data", "yyyyMMdd").cast("int"))
)


# =========================================================
# 10. Seleciona ordem final das colunas
# =========================================================

df_cal_final = df_cal.select(
    "data",
    "data_int",
    "ano",
    "mes_num",
    "nome_mes",
    "nome_mes_3",
    "ano_mes",
    "ano_mes_num",
    "mes_ano_nome",
    "trimestre",
    "dia_mes",
    "dia_semana_num",
    "nome_dia_semana",
    "nome_dia_3",
    "semana_ano",
    "data_inicio_semana",
    "data_fim_semana",
    "data_inicio_mes",
    "data_fim_mes",
    "eh_feriado",
    "nome_feriado",
    "tipo_feriado",
    "eh_dia_util",
    "eh_emenda",
    "tipo_emenda"
)


# =========================================================
# 11. Grava como tabela Delta no Lakehouse
# =========================================================

df_cal_final.write.mode("overwrite").format("delta").saveAsTable("dim_calendario")

StatementMeta(, 3b1f4433-bf7a-4388-a8eb-26934144c6bc, 3, Finished, Available, Finished, False)